%md
## Gold model

```text
gold_layer
│
├── fact_sales
│     └── Grain: one row per sale transaction
│
├── dim_customer
│
├── dim_product
│
├── dim_store
│     └── SCD Type 2
│
├── dim_location
│
└── gold_run_control
# MAGIC
Retention:
  Gold fact_sales = latest 7 runs
  Audit table     = complete historical run history
```


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime



In [0]:

dbutils.widgets.text("catalog", "sales")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("retention_runs", "7")

CATALOG = dbutils.widgets.get("catalog").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()
GOLD_SCHEMA = dbutils.widgets.get("gold_schema").strip()
CURRENT_RUN_ID = dbutils.widgets.get("run_id").strip()
RETENTION_RUNS = int(dbutils.widgets.get("retention_runs"))

if RETENTION_RUNS < 1:
    raise ValueError("retention_runs must be >= 1")

print(f"Catalog        : {CATALOG}")
print(f"Silver schema  : {SILVER_SCHEMA}")
print(f"Gold schema    : {GOLD_SCHEMA}")
print(f"Run ID         : {CURRENT_RUN_ID or '[auto-detect]'}")
print(f"Retention runs : {RETENTION_RUNS}")



In [0]:

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.sales"

FACT_SALES = f"{CATALOG}.{GOLD_SCHEMA}.fact_sales"
DIM_PRODUCT = f"{CATALOG}.{GOLD_SCHEMA}.dim_product"
DIM_CUSTOMER = f"{CATALOG}.{GOLD_SCHEMA}.dim_customer"
DIM_STORE = f"{CATALOG}.{GOLD_SCHEMA}.dim_store"
DIM_LOCATION = f"{CATALOG}.{GOLD_SCHEMA}.dim_location"
RUN_CONTROL = f"{CATALOG}.{GOLD_SCHEMA}.gold_run_control"



In [0]:

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_CONTROL}
(
    run_id STRING,
    processed_timestamp TIMESTAMP,
    record_count BIGINT,
    status STRING,
    message STRING
)
USING DELTA
""")



In [0]:

if not spark.catalog.tableExists(SILVER_TABLE):
    raise Exception(f"Silver table does not exist: {SILVER_TABLE}")

silver_df = spark.table(SILVER_TABLE)

print(f"Silver record count: {silver_df.count()}")

required_columns = [
    "sale_id",
    "run_id",
    "product_id",
    "product_name",
    "category",
    "subcategory",
    "customer_id",
    "customer_type",
    "store_id",
    "store_name",
    "country",
    "region",
    "state",
    "city",
    "zip_code",
    "sale_timestamp",
    "sale_date",
    "quantity",
    "unit_price",
    "gross_amount",
    "discount_amount",
    "net_amount_before_tax",
    "tax_amount",
    "total_amount",
    "shipping_cost",
    "cost_price",
    "profit_amount",
    "return_quantity",
    "return_flag",
    "currency",
    "payment_method",
    "sales_channel",
    "order_status",
    "data_source"
]

missing_columns = [
    c for c in required_columns
    if c not in silver_df.columns
]

if missing_columns:
    raise Exception(
        f"Missing required Silver columns: {missing_columns}"
    )



In [0]:

if not CURRENT_RUN_ID:
    latest_run = (
        silver_df
        .select("run_id")
        .where(F.col("run_id").isNotNull())
        .distinct()
        .orderBy(F.col("run_id").desc())
        .first()
    )

    if latest_run is None:
        raise Exception("No run_id found in Silver.")

    CURRENT_RUN_ID = latest_run["run_id"]

print(f"Current run: {CURRENT_RUN_ID}")



In [0]:
%skip
display(silver_df)

%md
## 6. Idempotency check
# MAGIC
The same run must not be loaded into Gold twice.



In [0]:

already_processed = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {RUN_CONTROL}
WHERE run_id = '{CURRENT_RUN_ID}'
  AND status = 'SUCCESS'
""").first()["cnt"]

if already_processed > 0:
    raise Exception(
        f"Run {CURRENT_RUN_ID} has already been successfully processed."
    )



%md
## 7. Extract current Silver run



In [0]:

current_run_df = (
    silver_df
    .filter(F.col("run_id") == CURRENT_RUN_ID)
)

current_count = current_run_df.count()

if current_count == 0:
    raise Exception(
        f"No Silver records found for run_id={CURRENT_RUN_ID}"
    )

print(f"Current run record count: {current_count}")



%md
## 8. Remove duplicate sale_id within current run
# MAGIC
`sale_id` is treated as the transaction business key.



In [0]:

dedup_window = (
    Window
    .partitionBy("sale_id")
    .orderBy(
        F.col("sale_timestamp").desc_nulls_last()
    )
)

current_run_df = (
    current_run_df
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

dedup_count = current_run_df.count()

print(f"Records after sale_id deduplication: {dedup_count}")



In [0]:

fact_current = (
    current_run_df
    .select(
        "sale_id",
        "run_id",

        "product_id",
        "customer_id",
        "store_id",

        "country",
        "region",
        "state",
        "city",
        "zip_code",

        "sale_timestamp",
        "sale_date",

        "quantity",
        "unit_price",
        "gross_amount",
        "discount_amount",
        "net_amount_before_tax",
        "tax_amount",
        "total_amount",

        "shipping_cost",
        "cost_price",
        "profit_amount",

        "return_quantity",
        "return_flag",

        "currency",
        "payment_method",
        "sales_channel",
        "order_status",
        "data_source"
    )
    .withColumn("product_key", F.xxhash64("product_id"))
    .withColumn("customer_key", F.xxhash64("customer_id"))
    .withColumn("store_key", F.xxhash64("store_id"))
    .withColumn(
        "location_key",
        F.xxhash64(
            "country",
            "state",
            "city",
            "zip_code"
        )
    )
    .withColumn(
        "sale_key",
        F.xxhash64("sale_id")
    )
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "gold_processed_date",
        F.current_date()
    )
)

display(fact_current)

%md
## 10. Create / append fact_sales
# MAGIC
New runs are appended. Existing runs are never overwritten.



In [0]:

if not spark.catalog.tableExists(FACT_SALES):

    (
        fact_current
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FACT_SALES)
    )

    print(f"Created {FACT_SALES}")

else:

    # Safety check: sale_id from the incoming run should not already exist.
    existing_sale_count = (
        spark.table(FACT_SALES)
        .join(
            fact_current.select("sale_id").distinct(),
            "sale_id",
            "inner"
        )
        .limit(1)
        .count()
    )

    if existing_sale_count > 0:
        raise Exception(
            f"At least one sale_id from run {CURRENT_RUN_ID} "
            f"already exists in Gold fact_sales."
        )

    (
        fact_current
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(FACT_SALES)
    )

    print(f"Appended run {CURRENT_RUN_ID} to {FACT_SALES}")



## 11. Verify current run in Gold

In [0]:

gold_current_count = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {FACT_SALES}
WHERE run_id = '{CURRENT_RUN_ID}'
""").first()["cnt"]

if gold_current_count != dedup_count:
    raise Exception(
        f"Gold verification failed. "
        f"Expected={dedup_count}, Found={gold_current_count}"
    )

print(
    f"Gold verification successful: "
    f"{CURRENT_RUN_ID} -> {gold_current_count} records"
)



## 12. Determine latest 7 runs
# MAGIC
The retention logic is based on the run's Gold processing timestamp.



In [0]:

run_history_df = spark.sql(f"""
SELECT
    run_id,
    MAX(gold_processed_timestamp) AS processed_timestamp,
    COUNT(*) AS record_count
FROM {FACT_SALES}
GROUP BY run_id
ORDER BY processed_timestamp DESC
""")

all_runs = [
    r["run_id"]
    for r in run_history_df.collect()
]

retained_runs = [
    r["run_id"]
    for r in run_history_df.limit(RETENTION_RUNS).collect()
]

runs_to_delete = [
    run_id
    for run_id in all_runs
    if run_id not in retained_runs
]

print("All Gold runs:")
for run_id in all_runs:
    print(f"  {run_id}")

print("\nRetained runs:")
for run_id in retained_runs:
    print(f"  {run_id}")

print("\nRuns to delete:")
for run_id in runs_to_delete:
    print(f"  {run_id}")



## 13. Delete runs older than the latest 7

In [0]:

for old_run_id in runs_to_delete:

    print(f"Deleting old Gold run: {old_run_id}")

    spark.sql(f"""
    DELETE FROM {FACT_SALES}
    WHERE run_id = '{old_run_id}'
    """)

print("Gold retention cleanup completed.")



## 14. Get retained Silver data
# MAGIC
Dimensions are modeled from the same 7-run business window as Gold fact data.



In [0]:

retained_runs_df = (
    spark.table(FACT_SALES)
    .select("run_id")
    .distinct()
)

retained_silver = (
    silver_df
    .join(
        retained_runs_df,
        on="run_id",
        how="inner"
    )
)

print(
    f"Retained Silver records used for dimensional modeling: "
    f"{retained_silver.count()}"
)



## 15. DIM PRODUCT
# MAGIC
One row per `product_id`.

In [0]:

dim_product = (
    retained_silver
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "cost_price"
    )
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_key",
        F.xxhash64("product_id")
    )
    .withColumn(
        "is_active",
        F.lit(True)
    )
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)

(
    dim_product
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_PRODUCT)
)

print(f"Created/updated {DIM_PRODUCT}")

display(dim_product)

## 16. DIM CUSTOMER
# MAGIC
One row per `customer_id`, with metrics calculated over the retained 7-run window.



In [0]:

dim_customer = (
    retained_silver
    .groupBy("customer_id")
    .agg(
        F.max_by(
            "customer_type",
            "sale_timestamp"
        ).alias("customer_type"),

        F.max_by(
            "country",
            "sale_timestamp"
        ).alias("country"),

        F.max_by(
            "region",
            "sale_timestamp"
        ).alias("region"),

        F.max_by(
            "state",
            "sale_timestamp"
        ).alias("state"),

        F.max_by(
            "city",
            "sale_timestamp"
        ).alias("city"),

        F.min("sale_date").alias(
            "first_purchase_date"
        ),

        F.max("sale_date").alias(
            "last_purchase_date"
        ),

        F.countDistinct("sale_id").alias(
            "total_orders"
        ),

        F.round(
            F.sum("total_amount"),
            2
        ).alias(
            "total_spend"
        )
    )
    .withColumn(
        "customer_key",
        F.xxhash64("customer_id")
    )
    .withColumn(
        "is_active",
        F.lit(True)
    )
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)

(
    dim_customer
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_CUSTOMER)
)

print(f"Created/updated {DIM_CUSTOMER}")

display(dim_customer)

## 17. DIM LOCATION
# MAGIC
One row per geographic combination.

In [0]:

dim_location = (
    retained_silver
    .select(
        "country",
        "region",
        "state",
        "city",
        "zip_code"
    )
    .dropDuplicates()
    .withColumn(
        "location_key",
        F.xxhash64(
            "country",
            "state",
            "city",
            "zip_code"
        )
    )
    .withColumn(
        "gold_processed_timestamp",
        F.current_timestamp()
    )
)

(
    dim_location
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_LOCATION)
)

print(f"Created/updated {DIM_LOCATION}")

display(dim_location)

%md
## 18. DIM STORE - SCD Type 2
# MAGIC
Business key: `store_id`
# MAGIC
If tracked store attributes change, the old version is closed and a new version is inserted.
# MAGIC
SCD2 columns:
- `store_key`
- `store_id`
- `store_name`
- `store_type`
- location attributes
- `effective_from`
- `effective_to`
- `is_current`



In [0]:

store_source = (
    retained_silver
    .select(
        "store_id",
        "store_name",
        "subcategory",
        "country",
        "region",
        "state",
        "city",
        "zip_code",
    )
    .withColumn(
        "_source_order",
        F.monotonically_increasing_id()
    )
)

store_window = (
    Window
    .partitionBy("store_id")
    .orderBy(F.col("_source_order").desc())
)

store_source = (
    store_source
    .withColumn(
        "_rn",
        F.row_number().over(store_window)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_source_order")
)
display(store_source)


### Create SCD2 dimension if it does not exist


In [0]:

if not spark.catalog.tableExists(DIM_STORE):

    dim_store_initial = (
        store_source
        .withColumn(
            "store_key",
            F.xxhash64(
                "store_id",
                F.lit(CURRENT_RUN_ID)
            )
        )
        .withColumn(
            "effective_from",
            F.current_date()
        )
        .withColumn(
            "effective_to",
            F.lit("9999-12-31").cast("date")
        )
        .withColumn(
            "is_current",
            F.lit(True)
        )
        .withColumn(
            "gold_processed_timestamp",
            F.current_timestamp()
        )
    )

    (
        dim_store_initial
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DIM_STORE)
    )

    print(f"Created initial SCD2 table: {DIM_STORE}")

# Apply SCD2 changes

else:

    existing_store = spark.table(DIM_STORE)

    current_store = (
        existing_store
        .filter(F.col("is_current") == True)
    )

    tracked_columns = [
        "store_name",
        "subcategory",
        "country",
        "region",
        "state",
        "city",
        "zip_code",
    ]

    comparison = (
        store_source.alias("src")
        .join(
            current_store.alias("tgt"),
            F.col("src.store_id") == F.col("tgt.store_id"),
            "left"
        )
    )

    change_condition = F.lit(False)

    for c in tracked_columns:
        change_condition = (
            change_condition |
            ~(
                F.col(f"src.{c}").eqNullSafe(
                    F.col(f"tgt.{c}")
                )
            )
        )

    changed_stores = (
        comparison
        .filter(
            F.col("tgt.store_id").isNotNull() &
            change_condition
        )
        .select(
            "src.*"
        )
    )

    new_stores = (
        comparison
        .filter(
            F.col("tgt.store_id").isNull()
        )
        .select(
            "src.*"
        )
    )

    changed_or_new = (
        changed_stores
        .unionByName(new_stores)
        .dropDuplicates(["store_id"])
    )

    changed_count = changed_stores.count()
    new_count = new_stores.count()

    print(f"New stores: {new_count}")
    print(f"Changed stores: {changed_count}")



### Close old SCD2 versions

In [0]:

if spark.catalog.tableExists(DIM_STORE):

    existing_store_delta = DeltaTable.forName(
        spark,
        DIM_STORE
    )

    changed_store_ids = (
        changed_stores
        .select("store_id")
        .distinct()
        .collect()
    )

    for row in changed_store_ids:
        store_id_value = row["store_id"]

        existing_store_delta.update(
            condition=(
                f"store_id = '{store_id_value}' "
                f"AND is_current = true"
            ),
            set={
                "is_current": "false",
                "effective_to": "current_date()",
                "gold_processed_timestamp":
                    "current_timestamp()"
            }
        )



%md
### Insert new/current SCD2 versions



In [0]:

if spark.catalog.tableExists(DIM_STORE):

    if "changed_or_new" in locals() and changed_or_new.count() > 0:

        scd2_inserts = (
            changed_or_new
            .withColumn(
                "store_key",
                F.xxhash64(
                    "store_id",
                    F.lit(CURRENT_RUN_ID)
                )
            )
            .withColumn(
                "effective_from",
                F.current_date()
            )
            .withColumn(
                "effective_to",
                F.lit("9999-12-31").cast("date")
            )
            .withColumn(
                "is_current",
                F.lit(True)
            )
            .withColumn(
                "gold_processed_timestamp",
                F.current_timestamp()
            )
        )

        (
            scd2_inserts
            .select(
                "store_key",
                "store_id",
                "store_name",
                "subcategory",
                "country",
                "region",
                "state",
                "city",
                "zip_code",
                "effective_from",
                "effective_to",
                "is_current",
                "gold_processed_timestamp"
            )
            .write
            .format("delta")
            .mode("append")
            .saveAsTable(DIM_STORE)
        )

        print(
            f"Inserted {scd2_inserts.count()} SCD2 store versions."
        )
    else:
        print("No new or changed stores.")



## 19. Final Gold validation

In [0]:

final_fact_count = spark.table(FACT_SALES).count()

final_run_count = (
    spark.table(FACT_SALES)
    .select("run_id")
    .distinct()
    .count()
)

print(f"Final fact_sales records : {final_fact_count}")
print(f"Final Gold run count     : {final_run_count}")

if final_run_count > RETENTION_RUNS:
    raise Exception(
        f"Retention validation failed. "
        f"Gold contains {final_run_count} runs, "
        f"expected <= {RETENTION_RUNS}."
    )



## 20. Record successful Gold run

In [0]:

spark.sql(f"""
INSERT INTO {RUN_CONTROL}
VALUES
(
    '{CURRENT_RUN_ID}',
    current_timestamp(),
    {gold_current_count},
    'SUCCESS',
    'Gold fact loaded, 7-run retention applied, dimensions modeled.'
)
""")

print(
    f"Run {CURRENT_RUN_ID} completed successfully."
)



## 21. Gold run summary

In [0]:

spark.sql(f"""
SELECT
    run_id,
    COUNT(*) AS record_count,
    MIN(sale_date) AS min_sale_date,
    MAX(sale_date) AS max_sale_date
FROM {FACT_SALES}
GROUP BY run_id
ORDER BY MAX(gold_processed_timestamp) DESC
""").show(truncate=False)

